# Model 2: Residual 3D CNN

Improved pipeline for classifying cognitive status (CN, MCI, or AD) from a single 3D T1-weighted brain MRI volume at each target visit (baseline, 12 months, and 24 months). Compared with the simple CNN in `01_simple_3d_cnn.ipynb`, it adds brain-masked intensity normalization, data augmentation, tempered class-balanced sampling, and a deeper residual 3D CNN with squeeze-and-excitation blocks.

> **Data notice:** The data used for this project are restricted and are not included in this repository. This notebook has been cleared of all references to the data and of all outputs. See `DATA_NOTICE.md`, and `README.md` for the expected input format.

### Load Data

This section defines the data paths, reads the labels table, maps diagnosis labels into three classes, and builds metadata linking each available MRI file to the diagnosis at a selected visit. Point `IMAGE_DIR` and `LABELS_PATH` to your own data.

In [ ]:
from pathlib import Path
from datetime import datetime

import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, classification_report
from scipy.ndimage import zoom

from tqdm.auto import tqdm

USE_PROGRESS_BAR = False


In [ ]:
# Paths to your own data. The data used for this project are restricted and are not
# included in this repository (see DATA_NOTICE.md).
DATA_ROOT = Path("./data")
IMAGE_DIR = DATA_ROOT / "images"        # one preprocessed 3D NIfTI volume per subject: <subject_id>.nii.gz
LABELS_PATH = DATA_ROOT / "labels.csv"  # one row per subject and visit

# Column names in the labels table.
SUBJECT_COL = "subject_id"
VISIT_COL = "visit"
DIAGNOSIS_COL = "diagnosis"

labels_table = pd.read_csv(LABELS_PATH, dtype={SUBJECT_COL: str}, low_memory=False)

VALID_TARGET_VISITS = ("baseline", "month12", "month24")
SELECTED_TARGET_VISIT = "baseline"

label_map = {
    "CN": 0,
    "MCI": 1,
    "Dementia": 2,
    "AD": 2,
}
class_names = ["CN", "MCI", "AD"]


def build_metadata_for_target(target_visit):
    """Build image metadata and class labels for one outcome visit."""
    if target_visit not in VALID_TARGET_VISITS:
        raise ValueError(f"target_visit must be one of {VALID_TARGET_VISITS}, got {target_visit!r}")

    label_df = labels_table[labels_table[VISIT_COL] == target_visit][[SUBJECT_COL, DIAGNOSIS_COL]].copy()
    label_df = label_df.dropna(subset=[DIAGNOSIS_COL])
    label_df["label"] = label_df[DIAGNOSIS_COL].map(label_map)
    label_df = label_df.dropna(subset=["label"])
    label_df["label"] = label_df["label"].astype(int)

    rows = []
    for f in sorted(IMAGE_DIR.glob("*.nii.gz")):
        subject_id = f.name[: -len(".nii.gz")]
        rows.append({SUBJECT_COL: subject_id, "image_path": f})

    image_df = pd.DataFrame(rows, columns=[SUBJECT_COL, "image_path"])
    metadata = image_df.merge(label_df[[SUBJECT_COL, "label", DIAGNOSIS_COL]], on=SUBJECT_COL, how="inner")
    metadata["target_visit"] = target_visit
    metadata["class_name"] = metadata["label"].map(dict(enumerate(class_names)))
    return metadata


metadata = build_metadata_for_target(SELECTED_TARGET_VISIT)
metadata.head(), metadata[DIAGNOSIS_COL].value_counts()

### EDA and Baseline Checks

Before changing the network, check whether the task is being limited by class imbalance, scan geometry, or intensity scaling. The next cells make those issues visible and motivate the fixes below.

In [ ]:
metadata_by_visit = {
    target_visit: build_metadata_for_target(target_visit)
    for target_visit in VALID_TARGET_VISITS
}

label_summary = (
    pd.concat(metadata_by_visit.values(), ignore_index=True)
    .groupby(["target_visit", "class_name"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=VALID_TARGET_VISITS, columns=class_names)
)

baseline_summary = label_summary.copy()
baseline_summary["majority_class"] = label_summary.idxmax(axis=1)
baseline_summary["majority_accuracy"] = label_summary.max(axis=1) / label_summary.sum(axis=1)
baseline_summary


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
label_summary.plot(kind="bar", ax=ax)
ax.set_title("Class balance by prediction visit")
ax.set_xlabel("Target visit")
ax.set_ylabel("Number of labeled MRI scans")
ax.legend(title="Class")
plt.xticks(rotation=0)
plt.tight_layout()

sample_scan_paths = metadata["image_path"].sample(
    n=min(60, len(metadata)),
    random_state=42
)
scan_stats = []
for image_path in sample_scan_paths:
    arr = nib.load(image_path).get_fdata(dtype=np.float32)
    finite_arr = np.nan_to_num(arr)
    brain = finite_arr[np.abs(finite_arr) > 1e-6]
    scan_stats.append({
        "shape": finite_arr.shape,
        "min": float(np.min(finite_arr)),
        "max": float(np.max(finite_arr)),
        "mean_nonzero": float(brain.mean()) if brain.size else 0.0,
        "std_nonzero": float(brain.std()) if brain.size else 0.0,
        "nonzero_fraction": float(brain.size / finite_arr.size),
    })

scan_stats_df = pd.DataFrame(scan_stats)
display(scan_stats_df.describe(include="all"))
scan_stats_df["shape"].value_counts().head()


### Dataset and Splits

`NiftiMRIDataset` loads one NIfTI MRI volume at a time, resizes it to a common shape, normalizes the non-background voxels, and returns a tensor-label pair. The split function keeps class proportions similar across train, validation, and test sets.


In [ ]:
class NiftiMRIDataset(Dataset):
    def __init__(self, metadata, target_shape=(64, 64, 64), augment=False):
        self.metadata = metadata.reset_index(drop=True)
        self.target_shape = target_shape
        self.augment = augment

    def __len__(self):
        return len(self.metadata)

    def _resize(self, arr):
        factors = [
            self.target_shape[0] / arr.shape[0],
            self.target_shape[1] / arr.shape[1],
            self.target_shape[2] / arr.shape[2],
        ]
        return zoom(arr, factors, order=1)

    def _normalize_brain(self, arr):
        # Normalizing over the whole cube lets the zero background dominate.
        # Use the non-background voxels for the MRI intensity scale and keep background at 0.
        mask = np.abs(arr) > 1e-6
        if mask.sum() < 100:
            mean = arr.mean()
            std = arr.std()
            return (arr - mean) / (std + 1e-6)

        brain = arr[mask]
        mean = brain.mean()
        std = brain.std()
        arr = (arr - mean) / (std + 1e-6)
        arr[~mask] = 0.0
        return arr

    def _augment(self, arr):
        if np.random.rand() < 0.5:
            arr = np.flip(arr, axis=0).copy()
        if np.random.rand() < 0.5:
            arr = np.flip(arr, axis=1).copy()
        if np.random.rand() < 0.25:
            arr = arr * np.random.uniform(0.9, 1.1)
        if np.random.rand() < 0.25:
            arr = arr + np.random.normal(0, 0.03, size=arr.shape).astype(np.float32)
        return arr

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]

        img = nib.load(row["image_path"])
        arr = img.get_fdata(dtype=np.float32)
        arr = np.nan_to_num(arr)
        arr = self._resize(arr)
        arr = self._normalize_brain(arr)

        if self.augment:
            arr = self._augment(arr)

        x = torch.from_numpy(arr.astype(np.float32)).unsqueeze(0)
        y = torch.tensor(row["label"], dtype=torch.long)

        return x, y


def make_splits(metadata, random_state=42):
    train_df, temp_df = train_test_split(
        metadata,
        test_size=0.30,
        random_state=random_state,
        stratify=metadata["label"]
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=1/3,
        random_state=random_state,
        stratify=temp_df["label"]
    )

    return train_df, val_df, test_df


train_df, val_df, test_df = make_splits(metadata)

print(len(train_df), len(val_df), len(test_df))
print(train_df["label"].value_counts())
print(val_df["label"].value_counts())
print(test_df["label"].value_counts())


### DataLoaders and Sampling

The DataLoaders wrap the MRI datasets for minibatch training. The training loader uses a tempered weighted sampler so minority diagnoses appear more often without fully flattening the class distribution. Validation and test loaders keep the original held-out distributions.


In [ ]:
batch_size = 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def make_class_weights(train_df, num_classes=3):
    counts = train_df["label"].value_counts().reindex(range(num_classes), fill_value=0).sort_index()
    weights = counts.sum() / (num_classes * counts.clip(lower=1))
    return torch.tensor(weights.to_numpy(dtype=np.float32), dtype=torch.float32)


def make_weighted_sampler(train_df, num_classes=3):
    class_weights = make_class_weights(train_df, num_classes=num_classes)
    # Full inverse-frequency sampling overcorrected and made MCI under-predicted.
    # Square-root tempering gives minority classes more exposure, but less aggressively.
    tempered_class_weights = torch.sqrt(class_weights)
    sample_weights = train_df["label"].map(lambda label: tempered_class_weights[int(label)].item()).to_numpy()
    return WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True
    )


def make_dataloaders(train_df, val_df, test_df, batch_size=8, target_shape=(64, 64, 64), num_workers=2):
    train_dataset = NiftiMRIDataset(train_df, target_shape=target_shape, augment=True)
    val_dataset = NiftiMRIDataset(val_df, target_shape=target_shape, augment=False)
    test_dataset = NiftiMRIDataset(test_df, target_shape=target_shape, augment=False)
    train_sampler = make_weighted_sampler(train_df)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=train_sampler,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda")
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda")
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda")
    )

    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_dataloaders(
    train_df,
    val_df,
    test_df,
    batch_size=batch_size
)

x_batch, y_batch = next(iter(train_loader))

print(x_batch.shape)
print(y_batch.shape)
print(y_batch)


### CNN Model

The model is a moderately larger 3D residual CNN. It uses GroupNorm for small batches, squeeze-excitation blocks for channel attention, and both average and max global pooling before classification.


In [ ]:
class SqueezeExcite3D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Conv3d(channels, hidden, kernel_size=1),
            nn.SiLU(inplace=True),
            nn.Conv3d(hidden, channels, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(self.pool(x))


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, groups=8, drop_path=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(min(groups, out_channels), out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(min(groups, out_channels), out_channels)
        self.se = SqueezeExcite3D(out_channels)
        self.act = nn.SiLU(inplace=True)
        self.dropout = nn.Dropout3d(drop_path) if drop_path > 0 else nn.Identity()

        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv3d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.GroupNorm(min(groups, out_channels), out_channels),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)
        x = self.act(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        x = self.se(x)
        x = self.dropout(x)
        x = self.act(x + identity)
        return x


class Small3DCNN(nn.Module):
    def __init__(self, num_classes=3, width=24, dropout=0.4):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv3d(1, width, kernel_size=5, stride=2, padding=2, bias=False),
            nn.GroupNorm(4, width),
            nn.SiLU(inplace=True),
        )

        self.features = nn.Sequential(
            ResidualBlock3D(width, width),
            ResidualBlock3D(width, width),
            ResidualBlock3D(width, width * 2, stride=2),
            ResidualBlock3D(width * 2, width * 2),
            ResidualBlock3D(width * 2, width * 4, stride=2, drop_path=0.05),
            ResidualBlock3D(width * 4, width * 4, drop_path=0.05),
            ResidualBlock3D(width * 4, width * 8, stride=2, drop_path=0.10),
            ResidualBlock3D(width * 8, width * 8, drop_path=0.10),
        )

        pooled_features = width * 8 * 2
        self.avg_pool = nn.AdaptiveAvgPool3d(1)
        self.max_pool = nn.AdaptiveMaxPool3d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(pooled_features),
            nn.Dropout(dropout),
            nn.Linear(pooled_features, width * 8),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(width * 8, width * 4),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(width * 4, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        x = torch.cat([self.avg_pool(x), self.max_pool(x)], dim=1)
        x = self.classifier(x)
        return x


### Training Setup

This section creates the model, cross-entropy loss with label smoothing, AdamW optimizer, and cosine learning-rate schedule. Class imbalance is handled by the weighted sampler defined above.

In [ ]:
def initialize_training(model_cls=Small3DCNN, num_classes=3, lr=3e-4, weight_decay=1e-3, train_df=None):
    model = model_cls(num_classes=num_classes).to(device)

    if train_df is None:
        class_weights = torch.ones(num_classes, dtype=torch.float32, device=device)
    else:
        class_weights = make_class_weights(train_df, num_classes=num_classes).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs if "num_epochs" in globals() else 50,
        eta_min=1e-6
    )
    return model, criterion, optimizer, scheduler


model, criterion, optimizer, scheduler = initialize_training(train_df=train_df)


In [ ]:
device

### Train and Evaluate Helpers

These helper functions contain the per-epoch training step and evaluation logic. They report accuracy, balanced accuracy, and macro F1 so performance can be judged beyond overall accuracy.


In [ ]:
def maybe_progress(loader, desc):
    if USE_PROGRESS_BAR:
        return tqdm(loader, desc=desc, leave=False)
    return loader


def train_one_epoch(model, loader, optimizer, criterion, device, desc="Training"):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for x, y in maybe_progress(loader, desc=desc):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    acc = accuracy_score(all_labels, all_preds)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, bal_acc, macro_f1


def evaluate(model, loader, criterion, device, desc="Evaluating"):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for x, y in maybe_progress(loader, desc=desc):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            logits = model(x)
            loss = criterion(logits, y)

            probs = F.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)

            total_loss += loss.item() * x.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    acc = accuracy_score(all_labels, all_preds)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "macro_f1": macro_f1,
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs),
    }


### Full Training Loop

The loop trains one model per target visit, tracks macro F1 on the validation set, saves the best checkpoint, and stops early if validation F1 stops improving. Macro F1 is emphasized because accuracy can hide poor minority-class performance.


In [ ]:
num_epochs = 10
early_stopping_patience = 10
training_started_at = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_root = Path("models") / training_started_at
checkpoint_root.mkdir(parents=True, exist_ok=True)

training_targets = list(VALID_TARGET_VISITS)
training_results = {}
all_histories = []

for target_visit in training_targets:
    print(f"\nTraining target_visit={target_visit}")

    target_metadata = build_metadata_for_target(target_visit)
    target_train_df, target_val_df, target_test_df = make_splits(target_metadata)
    target_train_loader, target_val_loader, target_test_loader = make_dataloaders(
        target_train_df,
        target_val_df,
        target_test_df,
        batch_size=batch_size
    )

    model, criterion, optimizer, scheduler = initialize_training(train_df=target_train_df)
    best_val_f1 = -1
    epochs_without_improvement = 0
    target_checkpoint_dir = checkpoint_root / target_visit
    target_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = target_checkpoint_dir / "best_3d_cnn.pt"
    history = []

    for epoch in range(num_epochs):
        train_loss, train_acc, train_bal_acc, train_f1 = train_one_epoch(
            model,
            target_train_loader,
            optimizer,
            criterion,
            device
        )

        val_results = evaluate(
            model,
            target_val_loader,
            criterion,
            device
        )

        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]

        row = {
            "target_visit": target_visit,
            "epoch": epoch + 1,
            "learning_rate": current_lr,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "train_balanced_accuracy": train_bal_acc,
            "train_macro_f1": train_f1,
            "val_loss": val_results["loss"],
            "val_accuracy": val_results["accuracy"],
            "val_balanced_accuracy": val_results["balanced_accuracy"],
            "val_macro_f1": val_results["macro_f1"],
        }

        history.append(row)
        all_histories.append(row)

        print(
            f"{target_visit} | "
            f"Epoch {epoch + 1:02d}/{num_epochs} | "
            f"LR: {current_lr:.2e} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train F1: {train_f1:.4f} | "
            f"Val Loss: {val_results['loss']:.4f} | "
            f"Val F1: {val_results['macro_f1']:.4f}"
        )

        if val_results["macro_f1"] > best_val_f1:
            best_val_f1 = val_results["macro_f1"]
            epochs_without_improvement = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"Saved best model to {best_model_path}")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stopping_patience:
            print(f"Early stopping after {epoch + 1} epochs without validation F1 improvement.")
            break

    training_results[target_visit] = {
        "metadata": target_metadata,
        "train_df": target_train_df,
        "val_df": target_val_df,
        "test_df": target_test_df,
        "train_loader": target_train_loader,
        "val_loader": target_val_loader,
        "test_loader": target_test_loader,
        "history": history,
        "best_val_f1": best_val_f1,
        "best_model_path": best_model_path,
    }

print(f"\nAll best models saved under: {checkpoint_root}")


### Training History Table

The history table gathers epoch-level metrics across all target visits. It is useful for checking whether the model is underfitting, overfitting, or reacting to the learning-rate schedule.


In [ ]:
history_df = pd.DataFrame(all_histories)
history_path = checkpoint_root / "training_history.csv"
history_df.to_csv(history_path, index=False)
print(f"Saved training history to {history_path}")
history_df


### Learning Curves

This plot compares train and validation macro F1 over time. A persistent gap suggests overfitting; both curves staying low suggests underfitting or weak input signal.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for target_visit, target_history in history_df.groupby("target_visit"):
    ax.plot(
        target_history["epoch"].to_numpy(dtype=float),
        target_history["train_macro_f1"].to_numpy(dtype=float),
        marker="o",
        linewidth=2,
        label=f"{target_visit} Train Macro F1"
    )
    ax.plot(
        target_history["epoch"].to_numpy(dtype=float),
        target_history["val_macro_f1"].to_numpy(dtype=float),
        marker="o",
        linewidth=2,
        linestyle="--",
        label=f"{target_visit} Validation Macro F1"
    )

ax.set_xlabel("Epoch")
ax.set_ylabel("Macro F1")
ax.set_title("Training Curves")
ax.set_ylim(0, 1)
ax.set_xticks(range(1, num_epochs + 1))
ax.grid(True, alpha=0.3)
ax.legend()

plt.show()


### Test Set Evaluation

After training, the best validation checkpoint for each target visit is reloaded and evaluated once on the held-out test set. This keeps model selection separate from final testing.


In [ ]:
test_results_by_target = {}

for target_visit, result in training_results.items():
    best_model = Small3DCNN(num_classes=3).to(device)
    best_model.load_state_dict(torch.load(result["best_model_path"], map_location=device, weights_only=True))
    test_criterion = nn.CrossEntropyLoss()

    test_results = evaluate(
        best_model,
        result["test_loader"],
        test_criterion,
        device
    )

    test_results_by_target[target_visit] = test_results

    print(f"\nTarget visit: {target_visit}")
    print("Best model path:", result["best_model_path"])
    print("Test Accuracy:", test_results["accuracy"])
    print("Test Balanced Accuracy:", test_results["balanced_accuracy"])
    print("Test Macro F1:", test_results["macro_f1"])

best_target_visit = max(
    training_results,
    key=lambda target_visit: training_results[target_visit]["best_val_f1"]
)
best_model_path = training_results[best_target_visit]["best_model_path"]
test_results = test_results_by_target[best_target_visit]

print(f"\nBest target by validation Macro F1: {best_target_visit}")


### Confusion Matrices and Class Reports

The confusion matrices show which diagnoses are being confused. This is especially important for MCI and AD, where overall accuracy can look acceptable while one class performs poorly.


In [ ]:
for target_visit, target_test_results in test_results_by_target.items():
    cm = confusion_matrix(
        target_test_results["labels"],
        target_test_results["preds"]
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"True {c}" for c in class_names],
        columns=[f"Pred {c}" for c in class_names]
    )

    print(f"\nTarget visit: {target_visit}")
    pred_counts = pd.Series(target_test_results["preds"]).map(dict(enumerate(class_names))).value_counts().reindex(class_names, fill_value=0)
    true_counts = pd.Series(target_test_results["labels"]).map(dict(enumerate(class_names))).value_counts().reindex(class_names, fill_value=0)
    print("True class counts:")
    print(true_counts)
    print("Predicted class counts:")
    print(pred_counts)
    print(cm_df)
    print(classification_report(
        target_test_results["labels"],
        target_test_results["preds"],
        target_names=class_names
    ))
